In [1]:
import pandas as pd


In [2]:
# load the ranking data

df = pd.read_csv("data/Lagos_Ranking_Data.csv")

In [3]:
# check if the data loaded correctly by displaying the first few rows of the dataframe

df.head()

,LGA,Ownership,Level,Learners,Learner_Teacher_Ratio,Learner_Classroom_Ratio,Learner_Toilet_Ratio,Female_Teachers_PCT,Special_Needs_Learners,LTR_Gap_vs_UBE35
0,Agege,Public,Primary,9529,28.0,16.3,21.5,82.4,12,-7.0
1,Ajeromi/Ifelodun,Public,Primary,13519,38.2,18.5,24.1,76.6,75,3.2
2,Alimosho,Public,Primary,30526,32.1,35.3,46.5,88.4,284,-2.9
3,Amuwo Odofin,Public,Primary,10313,34.6,19.7,28.5,70.8,98,-0.4
4,Apapa,Public,Primary,5559,27.9,25.1,19.2,68.8,63,-7.1


| Resource | UBE minimum standard | Meaning |
|---|---:|---|
| **Teacher** | **35 learners : 1 teacher** | 35 learners per teacher |
| **Classroom** | **35 learners : 1 classroom** | 35 learners per classroom |
| **Toilet** | **30 learners : 1 toilet** | 30 learners per toilet |

The key source for this scoring benchmark is the Federal Ministry of Education's current EMIS statistics portal.

In [6]:
# Calculate the gap for each resource by
# Comparing the actualresource ratios against UBE standard 

df["teacher_gap"] = ((df["Learner_Teacher_Ratio"] - 35) / 35).clip(lower=0)

df["classroom_gap"] = ((df["Learner_Classroom_Ratio"] - 35) / 35).clip(lower=0)

df["toilet_gap"] = ((df["Learner_Toilet_Ratio"] - 30) / 30).clip(lower=0)

In [7]:
# Check the calculated gaps
df[
    [
        "LGA",
        "Learner_Teacher_Ratio",
        "teacher_gap",
        "Learner_Classroom_Ratio",
        "classroom_gap",
        "Learner_Toilet_Ratio",
        "toilet_gap"
    ]
].head(10)

,LGA,Learner_Teacher_Ratio,teacher_gap,Learner_Classroom_Ratio,classroom_gap,Learner_Toilet_Ratio,toilet_gap
0,Agege,28.0,0.000000,16.3,0.000000,21.5,0.00
1,Ajeromi/Ifelodun,38.2,0.091429,18.5,0.000000,24.1,0.00
2,Alimosho,32.1,0.000000,35.3,0.008571,46.5,0.55
3,Amuwo Odofin,34.6,0.000000,19.7,0.000000,28.5,0.00
4,Apapa,27.9,0.000000,25.1,0.000000,19.2,0.00
5,Badagry,32.5,0.000000,38.1,0.088571,41.7,0.39
6,Epe,22.7,0.000000,22.7,0.000000,34.8,0.16
7,Eti-Osa,41.8,0.194286,33.3,0.000000,35.1,0.17
8,Ibeju/Lekki,39.4,0.125714,34.1,0.000000,51.9,0.73
9,Ifako/Ijaye,19.8,0.000000,23.1,0.000000,25.8,0.00


In [15]:
# Calculate the average gap across all 3 resources (teachers, classrooms, and toilets)

df["resource_gap_raw"] = (
    df["teacher_gap"] +
    df["classroom_gap"] +
    df["toilet_gap"]
) / 3

In [10]:
# Check the resource gap scores
df[["LGA", "resource_gap_raw", "resource_gap_score"]].head(10)


,LGA,resource_gap_raw,resource_gap_score
0,Agege,0.000000,0.000000
1,Ajeromi/Ifelodun,0.030476,1.527811
2,Alimosho,0.186190,9.333970
3,Amuwo Odofin,0.000000,0.000000
4,Apapa,0.000000,0.000000
5,Badagry,0.159524,7.997135
6,Epe,0.053333,2.673669
7,Eti-Osa,0.121429,6.087372
8,Ibeju/Lekki,0.285238,14.299355
9,Ifako/Ijaye,0.000000,0.000000


In [13]:
# Check the education levels in the data

df[["LGA", "Level", "resource_gap_score"]].sort_values(["LGA", "Level"])

,LGA,Level,resource_gap_score
20,Agege,JSS,34.407575
0,Agege,Primary,0.000000
40,Agege,SSS,18.166627
21,Ajeromi/Ifelodun,JSS,82.167582
1,Ajeromi/Ifelodun,Primary,1.527811
41,Ajeromi/Ifelodun,SSS,54.380520
22,Alimosho,JSS,53.091430
2,Alimosho,Primary,9.333970
42,Alimosho,SSS,44.974934
23,Amuwo Odofin,JSS,45.030636


In [16]:
# Combine the three school levels for each LGA

lga_scores = df.groupby("LGA")["resource_gap_raw"].mean().reset_index()

In [22]:
# Check the LGA scores

lga_scores.head(20)

,LGA,resource_gap_raw
0,Agege,0.349577
1,Ajeromi/Ifelodun,0.918095
2,Alimosho,0.714127
3,Amuwo Odofin,0.386720
4,Apapa,0.544074
5,Badagry,1.007884
6,Epe,0.313439
7,Eti-Osa,0.271587
8,Ibeju/Lekki,0.742275
9,Ifako/Ijaye,0.610899


In [23]:
# Convert the LGA resource gap to a score out of 100

lga_scores["resource_gap_score"] = (
    lga_scores["resource_gap_raw"] /
    lga_scores["resource_gap_raw"].max()
) * 100

In [24]:
# Check the LGA resource gap scores

lga_scores.head(20)

,LGA,resource_gap_raw,resource_gap_score
0,Agege,0.349577,34.684235
1,Ajeromi/Ifelodun,0.918095,91.091396
2,Alimosho,0.714127,70.854113
3,Amuwo Odofin,0.386720,38.369468
4,Apapa,0.544074,53.981836
5,Badagry,1.007884,100.000000
6,Epe,0.313439,31.098745
7,Eti-Osa,0.271587,26.946296
8,Ibeju/Lekki,0.742275,73.646911
9,Ifako/Ijaye,0.610899,60.612106


Ranking the LGAs

The LGA with the highest resource-gap score should be Rank 1.

In [25]:
# Rank the LGAs from highest to lowest resource gap
lga_scores["rank"] = lga_scores["resource_gap_score"].rank(
    ascending=False,
    method="min"
).astype(int)

In [26]:
# Display the final LGA ranking

lga_ranking = lga_scores.sort_values(
    "resource_gap_score",
    ascending=False
)

lga_ranking[
    ["rank", "LGA", "resource_gap_score"]
]

,rank,LGA,resource_gap_score
5,1,Badagry,100.000000
1,2,Ajeromi/Ifelodun,91.091396
16,3,Ojo,85.400808
8,4,Ibeju/Lekki,73.646911
2,5,Alimosho,70.854113
11,6,Ikorodu,69.095491
9,7,Ifako/Ijaye,60.612106
4,8,Apapa,53.981836
3,9,Amuwo Odofin,38.369468
18,10,Shomolu,36.952071


In [27]:
# Save the final LGA ranking file to a CSV for further analysis and reporting

lga_ranking.to_csv("data/Lagos_LGA_Resource_Gap_Ranking.csv", index=False)